# 2 Baseline 跑通

本章使用课程内置的 cann-recipes-infer 推理框架在 NPU 上运行 `Qwen3-8B` 离线推理。完成后会得到指标文件和生成文本文件，作为后续 Profiling 分析与优化 A/B 对比的基线。


## 1 环境与工作区准备
定位教程目录，创建本次实验的运行目录，并把 CANN 环境变量导入当前 Notebook 进程。第 2 章会首次检查 1-6 章统一依赖：仅当包缺失或版本不匹配时才安装或替换，全部满足时跳过安装；后续章节直接复用该环境。运行目录指 `Sources/model_inference_optimization/qwen3_8b`，用于保存 Notebook 生成的临时 YAML、推理日志、metrics、outputs 和后续 Profiling 结果；课程模板和源码在 `tutorials/llm_inference/qwen3_8b`。模型权重在执行推理前写入 recipes YAML；没有本地权重时，会先通过默认 ModelScope 通道获取 `Qwen/Qwen3-8B`。


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)
os.environ.setdefault('QWEN3_8B_DOWNLOAD_BACKEND', 'modelscope')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')

def locate_repo_root():
    repo_roots = []
    for env_name in ('GITCODE_REPO_ROOT', 'CANN_LEARNING_HUB_ROOT', 'REPO_ROOT'):
        if os.environ.get(env_name):
            repo_roots.append(Path(os.environ[env_name]))
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/qwen3_8b/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root. Open this notebook inside cann-learning-hub or set CANN_LEARNING_HUB_ROOT to the repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'qwen3_8b'
SRC_DIR = TUTORIAL_DIR / 'src'
SOURCE_DIR = REPO_ROOT / 'Sources' / 'model_inference_optimization' / 'qwen3_8b'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from inference_scripts.environment_check import find_unmet_requirements, read_pinned_requirements

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment before running this notebook.')
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

REQUIREMENTS_FILE = TUTORIAL_DIR / 'requirements.txt'

pinned_requirements = read_pinned_requirements(REQUIREMENTS_FILE)
unmet_requirements = find_unmet_requirements(pinned_requirements)
if unmet_requirements:
    print('[依赖检查] 以下依赖缺失或版本不匹配：')
    for name, actual, expected in unmet_requirements:
        print(f'  - {name}: 当前={actual or "未安装"}, 要求={expected}')
    requirements_to_install = [f'{name}=={expected}' for name, _, expected in unmet_requirements]
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install',
            '-i', 'https://pypi.tuna.tsinghua.edu.cn/simple',
            '--extra-index-url', 'https://download.pytorch.org/whl/cpu',
            '--trusted-host', 'pypi.tuna.tsinghua.edu.cn',
            *requirements_to_install,
        ],
        check=True,
    )
    remaining = find_unmet_requirements(pinned_requirements)
    if remaining:
        details = ', '.join(f'{name}: {actual or "未安装"} != {expected}' for name, actual, expected in remaining)
        raise RuntimeError(f'依赖安装后仍不满足统一版本：{details}')
    print('[依赖检查] 安装完成，统一环境已满足')
else:
    print('[依赖检查] 所有依赖版本均满足，跳过安装')

# 让后续 %%bash（infer.sh -> function.sh 里的 python3）使用与本内核相同的解释器。
# 不加这行时 PATH 上的 python3 可能是另一套环境，会缺 datasets 等依赖。
os.environ['PATH'] = str(Path(sys.executable).parent) + os.pathsep + os.environ['PATH']

RECIPE_ROOT = SRC_DIR / 'inference_scripts' / 'recipe_qwen3_8b'
RECIPE_CONFIG_DIR = RECIPE_ROOT / 'models' / 'qwen' / 'config'
BASELINE_YAML_TEMPLATE = RECIPE_CONFIG_DIR / 'qwen3_8b_1tp.yaml'
PROFILE_YAML_TEMPLATE = RECIPE_CONFIG_DIR / 'qwen3_8b_1tp_profile.yaml'
OPTIMIZED_YAML_TEMPLATE = RECIPE_CONFIG_DIR / 'qwen3_8b_1tp_add_rmsnorm.yaml'
INFER_SH = RECIPE_ROOT / 'executor' / 'scripts' / 'infer.sh'
MODEL_ID = os.environ.get('QWEN3_8B_MODEL_PATH') or 'Qwen/Qwen3-8B'
MODEL_SOURCE_HINT = 'local_path' if os.environ.get('QWEN3_8B_MODEL_PATH') else os.environ['QWEN3_8B_DOWNLOAD_BACKEND']

print('仓库目录 =', REPO_ROOT)
print('教程目录 =', TUTORIAL_DIR)
print('运行目录 =', SOURCE_DIR)
print('recipes 目录 =', RECIPE_ROOT)
print('YAML 目录 =', RECIPE_CONFIG_DIR)
print('推理入口 =', INFER_SH)
print('可见 NPU =', os.environ.get('ASCEND_RT_VISIBLE_DEVICES'))
print('Python 解释器 =', sys.executable)
print('[环境准备] 完成', flush=True)


## 2 查看 YAML 并执行 Baseline 推理

本节按 recipes 的常规流程执行 Baseline：先查看并准备 YAML，再通过 `infer.sh` 启动离线推理，最后读取 recipes 运行结果并生成课程指标。


### 2.1 准备 Baseline YAML

Qwen3-8B 单卡 YAML 位于 `models/qwen/config/qwen3_8b_1tp.yaml`。手工运行时，重点确认 `model_config.model_path`、`model_config.enable_profiler`、`custom_params.enable_npu_rmsnorm`、`custom_params.enable_add_rmsnorm` 和 `scheduler_config.max_new_tokens`。下面的 Python 单元会复制课程 YAML 到运行目录，写入当前模型路径，并打印关键字段。


In [ ]:
import os
import time
from inference_scripts.recipe_workflow import (
    collect_recipe_metrics,
    find_recipe_case_dir,
    prepare_runtime_yaml,
    print_yaml_focus,
    resolve_model_path,
)
from inference_scripts.inference_qwen3_8b import ensure_recipe_log_ok

BASELINE_WORK_DIR = SOURCE_DIR / 'run_outputs' / 'baseline_demo'
BASELINE_YAML = SOURCE_DIR / 'recipe_yaml' / 'baseline_demo.yaml'
BASELINE_WORK_DIR.mkdir(parents=True, exist_ok=True)
print('[Baseline 推理] 开始', flush=True)
resolved_model_path, model_source = resolve_model_path(str(MODEL_ID), quiet_model_io=False)
prepare_runtime_yaml(BASELINE_YAML_TEMPLATE, BASELINE_YAML, model_path=resolved_model_path)
print('[YAML 准备] Baseline 配置', flush=True)
print('参考 YAML =', BASELINE_YAML_TEMPLATE)
print('运行 YAML =', BASELINE_YAML)
print_yaml_focus(BASELINE_YAML)

baseline_log_dir = BASELINE_WORK_DIR / 'recipe_logs'
baseline_log_dir.mkdir(parents=True, exist_ok=True)
os.environ['QWEN3_RECIPE_ROOT'] = str(RECIPE_ROOT)
os.environ['QWEN3_RUN_YAML'] = str(BASELINE_YAML)
os.environ['QWEN3_RUN_LOG'] = str(baseline_log_dir / 'baseline_demo_infer_stdout.log')
os.environ['QWEN3_RUN_START_TIME'] = str(time.perf_counter())


### 2.2 使用 recipes 命令启动推理

这部分展示 recipes 的运行方式：先进入 `recipe_qwen3_8b` 工程目录，再通过 `executor/scripts/infer.sh` 加载 YAML 执行离线推理。代码中会显式打印并执行对应命令。


In [ ]:
%%bash
set -euo pipefail

# recipes 原生启动命令
echo 'recipes 启动命令:'
echo "cd ${QWEN3_RECIPE_ROOT}"
echo "bash executor/scripts/infer.sh --model qwen --mode offline --yaml ${QWEN3_RUN_YAML}"
cd "${QWEN3_RECIPE_ROOT}"
bash executor/scripts/infer.sh --model qwen --mode offline --yaml "${QWEN3_RUN_YAML}" 2>&1 | tee "${QWEN3_RUN_LOG}"


### 2.3 读取 recipes 运行结果

推理结束后，recipes 会在 `models/qwen/res/<日期>/<模型名>_<yaml名>/` 下生成日志。下面的单元读取 `log_0.log`，并整理为课程使用的 metrics 与生成文本文件。


In [ ]:
case_dir = find_recipe_case_dir(RECIPE_ROOT, BASELINE_YAML)
if case_dir is None:
    raise RuntimeError(f'未找到 recipes 结果目录：{BASELINE_YAML.stem}')
recipe_log_path = case_dir / 'log_0.log'
ensure_recipe_log_ok(recipe_log_path)
run_info = {
    'stdout_path': os.environ['QWEN3_RUN_LOG'],
    'stderr_path': os.environ['QWEN3_RUN_LOG'],
    'wall_time_s': time.perf_counter() - float(os.environ['QWEN3_RUN_START_TIME']),
    'recipe_case_dir': str(case_dir),
    'recipe_log_path': str(recipe_log_path),
}
baseline_metrics = collect_recipe_metrics(
    RECIPE_ROOT,
    BASELINE_WORK_DIR,
    'baseline_demo',
    BASELINE_YAML,
    run_info,
    model_id=str(MODEL_ID),
    resolved_model_path=resolved_model_path,
    model_source=model_source,
    optimization_mode='none',
    max_new_tokens=64,
    max_input_tokens=256,
    enable_thinking=False,
)
print('[Baseline 推理] 完成', flush=True)


## 3 查看结果

运行结束后，查看 Baseline 指标、生成文本和 recipes 结果目录。这里重点确认生成文本可读，Dense RMSNorm 融合状态为关闭。


In [ ]:
print('[结果查看] 开始', flush=True)
def status_cn(status):
    return {'ok': '通过', 'warning': '告警', 'failed': '失败'}.get(status, status)

def yes_no(value):
    if value is None:
        return '未记录'
    return '是' if value else '否'

def load_metrics(work_dir, tag):
    metrics_path = Path(work_dir) / 'metrics' / f'{tag}_summary.json'
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('指标文件 =', metrics_path)
    print('recipes YAML =', metrics.get('recipe_yaml_path'))
    print('recipes 结果目录 =', metrics.get('recipe_case_dir'))
    print('生成文件 =', metrics.get('outputs_path'))
    print('生成 token 数 =', metrics.get('generated_tokens'))
    print('总耗时(s) =', metrics.get('total_time_s'))
    print('吞吐(tokens/s) =', metrics.get('tokens_per_second'))
    rmsnorm_status = metrics.get('rmsnorm_status') or {}
    print('RMSNorm 融合 =', yes_no(rmsnorm_status.get('rmsnorm_fused')))
    print('Add+RMSNorm 融合 =', yes_no(rmsnorm_status.get('add_rmsnorm_fused')))
    sanity = metrics.get('generation_text_sanity')
    if sanity:
        print('吐字检查 =', status_cn(sanity.get('status')), f"警告={sanity.get('num_with_warnings')}", f"失败={sanity.get('num_failed')}")
    return metrics

baseline_metrics = load_metrics(BASELINE_WORK_DIR, 'baseline_demo')
print()
print('生成文本:')
for line in Path(baseline_metrics['outputs_path']).read_text(encoding='utf-8').splitlines():
    if line.strip():
        print(json.loads(line).get('generated_text', ''))
print('[结果查看] 完成', flush=True)


## 4（选做）动手试一试：自定义提示词

本节为选做内容，跳过不影响后续章节。Baseline 跑通后，可以换成你自己的问题试试：修改下方单元第一行的 `PROMPT`，运行该单元，即可在 NPU 上用 Qwen3-8B 生成回答并直接显示在下方。

本节沿用 recipes 原生命令 `infer.sh`：提示词写入独立数据目录，通过 YAML 的 `data_config.dataset_path` 传入；结果保存在 `run_outputs/custom_prompt`，不会覆盖 Baseline 的指标与生成文本。运行前需在本内核中先执行过第 1 节和 2.1 节（用到其中的环境变量与模型路径）；每次运行会重新加载模型，约需 1–2 分钟。


In [ ]:
PROMPT = '用一段话介绍一下CANN。'   # ← 在这里输入你的提示词
MAX_NEW_TOKENS = 256                                # 最多生成的 token 数，越大回答越长

import re
import yaml
from IPython.display import Markdown, display

if not PROMPT.strip():
    raise ValueError('PROMPT 为空，请先输入提示词')

# 1) 写入提示词与运行 YAML（独立目录，不影响 Baseline）
CUSTOM_WORK_DIR = SOURCE_DIR / 'run_outputs' / 'custom_prompt'
CUSTOM_DATASET_DIR = CUSTOM_WORK_DIR / 'dataset'
CUSTOM_YAML = SOURCE_DIR / 'recipe_yaml' / 'custom_prompt_demo.yaml'
CUSTOM_DATASET_DIR.mkdir(parents=True, exist_ok=True)
(CUSTOM_DATASET_DIR / 'default_prompt.json').write_text(
    json.dumps({'text': PROMPT.strip()}, ensure_ascii=False, indent=2), encoding='utf-8'
)
custom_config = prepare_runtime_yaml(BASELINE_YAML_TEMPLATE, CUSTOM_YAML, model_path=resolved_model_path)
custom_config.setdefault('data_config', {}).update(
    dataset='default', dataset_path=str(CUSTOM_DATASET_DIR), input_truncated_len=512
)
custom_config.setdefault('scheduler_config', {}).update(
    max_new_tokens=int(MAX_NEW_TOKENS), batch_size=1, enable_thinking=False
)
CUSTOM_YAML.write_text(yaml.safe_dump(custom_config, allow_unicode=True, sort_keys=False), encoding='utf-8')

# 2) 用 recipes 原生命令启动推理（完整日志写入文件，这里只显示进度）
custom_log_dir = CUSTOM_WORK_DIR / 'recipe_logs'
custom_log_dir.mkdir(parents=True, exist_ok=True)
stdout_log = custom_log_dir / 'custom_prompt_infer_stdout.log'
cmd = ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--mode', 'offline', '--yaml', str(CUSTOM_YAML)]
print('提示词 =', PROMPT.strip())
print('recipes 启动命令 = cd', RECIPE_ROOT, '&&', ' '.join(cmd))
print('推理中，正在加载模型并生成回答 ...', flush=True)
run_start = time.time()
with open(stdout_log, 'w', encoding='utf-8') as log_file:
    proc = subprocess.run(cmd, cwd=RECIPE_ROOT, env=os.environ.copy(), stdout=log_file, stderr=subprocess.STDOUT)
if proc.returncode != 0:
    print(''.join(stdout_log.read_text(encoding='utf-8', errors='replace').splitlines(True)[-30:]))
    raise RuntimeError(f'推理失败（返回码 {proc.returncode}），完整日志：{stdout_log}')

# 3) 从 recipes 日志读取完整回答并显示
custom_case_dir = find_recipe_case_dir(RECIPE_ROOT, CUSTOM_YAML)
custom_log_path = None if custom_case_dir is None else custom_case_dir / 'log_0.log'
if custom_log_path is None or not custom_log_path.is_file() or custom_log_path.stat().st_mtime < run_start:
    raise RuntimeError(f'未找到本次推理的 log_0.log，完整输出：{stdout_log}')
log_text = custom_log_path.read_text(encoding='utf-8', errors='replace')
match = re.search(r'Request\s+0:\s+outputs:\s?(.*?)(?=\n\[(?:INFO|WARNING|ERROR|DEBUG)\]\[|\Z)', log_text, re.S)
if match is None:
    raise RuntimeError(f'日志中没有找到生成结果：{custom_log_path}')
answer = match.group(1).strip()

answer_path = CUSTOM_WORK_DIR / 'outputs' / 'custom_prompt_answer.json'
answer_path.parent.mkdir(parents=True, exist_ok=True)
answer_path.write_text(json.dumps({'prompt': PROMPT.strip(), 'answer': answer, 'max_new_tokens': MAX_NEW_TOKENS,
                                   'recipe_log_path': str(custom_log_path)}, ensure_ascii=False, indent=2), encoding='utf-8')
decode_match = re.search(r'decode average inference time cost is\s+([0-9.]+)\s+ms', log_text)
print(f'推理完成，用时 {time.time() - run_start:.1f} s', end='')
print(f'，单步解码平均 {decode_match.group(1)} ms' if decode_match else '')
display(Markdown(f'**提示词：** {PROMPT.strip()}\n\n**Qwen3-8B 的回答：**\n\n{answer}'))
